# 2 · SFT (GPU DDP job)
Fine-tunes the base model on tool-call format; uploads the merged model to `{sft_model_s3}`. Entry point **`sft.sh`** → `torchrun --nproc_per_node=$NUM_GPUS sft_job.py` (real multi-GPU DDP).

In [ ]:
import json
from sagemaker.core import image_uris
from sagemaker.core.helper.session_helper import Session
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.train.configs import SourceCode, Compute, StoppingCondition

sess = Session()
cfg = json.load(open('toolcall_config.json'))
region = cfg['region']
INSTANCE_TYPE = cfg['gpu_instance']
image_uri = cfg['gpu_image_uri'] or image_uris.retrieve(
    framework='pytorch', region=region, version=cfg['pt_version'],
    instance_type=INSTANCE_TYPE, image_scope='training')
print('Region  :', region)
print('Instance:', INSTANCE_TYPE)
print('Image   :', image_uri)

In [ ]:
environment = {
    'BASE_MODEL': cfg['base_model'],
    'DATA_S3': cfg['data_s3'],
    'OUTPUT_SFT_S3': cfg['sft_model_s3'],
    'SEQ_LEN': str(cfg['seq_len']), 'SEED': str(cfg['seed']),
    'ATTN_IMPL': cfg['attn_impl'],
    'USE_LORA': str(cfg['use_lora']), 'LORA_R': str(cfg['lora_r']), 'LORA_ALPHA': str(cfg['lora_alpha']),
    'SFT_EPOCHS': str(cfg['sft_epochs']), 'SFT_LR': str(cfg['sft_lr']),
    'SFT_BS': str(cfg['sft_bs']), 'SFT_GRAD_ACCUM': str(cfg['sft_grad_accum']),
    'NUM_GPUS': str(cfg['num_gpus']),
    'HF_HOME': '/opt/ml/hf_home', 'HF_HUB_ENABLE_HF_TRANSFER': '1',
    'TOKENIZERS_PARALLELISM': 'false', 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
}
if cfg.get('hf_token'):
    environment['HF_TOKEN'] = cfg['hf_token']
    environment['HUGGING_FACE_HUB_TOKEN'] = cfg['hf_token']
source_code = SourceCode(source_dir='../src', requirements='requirements.txt', entry_script='sft.sh')
trainer = ModelTrainer(sagemaker_session=sess, role=cfg['role'], training_image=image_uri,
    source_code=source_code,
    compute=Compute(instance_type=INSTANCE_TYPE, instance_count=1, keep_alive_period_in_seconds=3600),
    environment=environment, base_job_name='toolcall-sft',
    stopping_condition=StoppingCondition(max_runtime_in_seconds=24*3600))
trainer.train(wait=True)
print('SFT model ->', cfg['sft_model_s3'])

Expect `launching torchrun with 8 processes (DDP)` in the log; only rank 0 merges + uploads.

**Next:** `03_grpo.ipynb`